### Environment Setup & Groq/LangSmith Client Initialization

In [2]:
# %% [markdown]
# ### Cell 1: Environment Setup & API Connection

# %%
import os
from dotenv import load_dotenv
from openai import OpenAI
from langsmith import Client

# ---------------------------------------------------------------------------
# 1. Load keys from .env
# ---------------------------------------------------------------------------
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("❌ GROQ_API_KEY is missing from your .env file!")
if not LANGCHAIN_API_KEY:
    raise ValueError("❌ LANGCHAIN_API_KEY is missing from your .env file!")

# Set environment variables for LangChain / LangSmith
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://apac.api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
os.environ["LANGCHAIN_PROJECT"] = "Retail-Chunking-Experiments"

MODEL_NAME = "openai/gpt-oss-20b"

# ---------------------------------------------------------------------------
# 2. Initialize Clients (Passing API Keys explicitly)
# ---------------------------------------------------------------------------
# OpenAI-compatible Client for Groq
groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=GROQ_API_KEY,
)

# Pass the API key explicitly to avoid environment resolution bugs
ls_client = Client(api_key=LANGCHAIN_API_KEY)

print("✅ Groq OpenAI-compatible API Client connected.")
print(f" Target Model: {MODEL_NAME}")
print("✅ LangSmith Client connected.")

✅ Groq OpenAI-compatible API Client connected.
 Target Model: openai/gpt-oss-20b
✅ LangSmith Client connected.


### Synthetic Ground-Truth Dataset Generation (Ragas + Groq)

In [3]:
# %% [markdown]
# ### Cell 2: Synthetic Ground-Truth Dataset Generation (Ragas + Groq)

# %%
import sys
import types
import os
import glob

# ---------------------------------------------------------------------------
# 1. Compatibility Patch: Bypass Ragas legacy VertexAI import issue
# ---------------------------------------------------------------------------
try:
    import langchain_community.chat_models.vertexai
except ModuleNotFoundError:
    dummy_module = types.ModuleType("langchain_community.chat_models.vertexai")
    dummy_module.ChatVertexAI = type("ChatVertexAI", (object,), {})
    sys.modules["langchain_community.chat_models.vertexai"] = dummy_module

from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# ---------------------------------------------------------------------------
# 2. Load Document Corpus from 'document_corpus' folder
# ---------------------------------------------------------------------------
docs_folder = "document_corpus"
documents = []

if os.path.exists(docs_folder):
    for filepath in glob.glob(f"{docs_folder}/*.*"):
        if filepath.endswith(".pdf"):
            loader = PyPDFLoader(filepath)
            documents.extend(loader.load())
        elif filepath.endswith(".txt") or filepath.endswith(".md"):
            loader = TextLoader(filepath, encoding="utf-8")
            documents.extend(loader.load())
    print(f" Loaded {len(documents)} document pages/files from '{docs_folder}'.")
else:
    raise FileNotFoundError(f"❌ '{docs_folder}' folder does not exist. Create it and place your source documents inside.")

if len(documents) == 0:
    raise ValueError(f"❌ No .pdf, .txt, or .md files found inside '{docs_folder}'.")

# Assign 'filename' metadata tag required by Ragas
for doc in documents:
    if "filename" not in doc.metadata:
        doc.metadata["filename"] = doc.metadata.get("source", "document.txt")

# ---------------------------------------------------------------------------
# 3. Configure Ragas Synthesizer Engine (Ragas v0.2+ Compatible)
# ---------------------------------------------------------------------------
langchain_llm = ChatOpenAI(
    model=MODEL_NAME,
    openai_api_key=GROQ_API_KEY,
    openai_api_base="https://api.groq.com/openai/v1",
    temperature=0.3
)

langchain_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

generator_llm = LangchainLLMWrapper(langchain_llm)
generator_embeddings = LangchainEmbeddingsWrapper(langchain_embeddings)

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings
)

# ---------------------------------------------------------------------------
# 4. Synthesize Ground-Truth Dataset & Push to LangSmith
# ---------------------------------------------------------------------------
DATASET_NAME = "Chunking_Strategy_Benchmark_v1"

if not ls_client.has_dataset(dataset_name=DATASET_NAME):
    print("⚡ Synthesizing multi-hop benchmark Q&A pairs from documents via Ragas Knowledge Graph...")
    
    # Generate 15 synthetic evaluation pairs
    testset = generator.generate_with_langchain_docs(documents, testset_size=15)
    synth_df = testset.to_pandas()
    
    # Create LangSmith Dataset
    dataset = ls_client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Synthetically generated benchmark Q&A pairs from local document corpus."
    )
    
    # Map questions and ground truths safely to LangSmith schema
    inputs = [
        {"question": row.get("user_input", row.get("question", ""))} 
        for _, row in synth_df.iterrows()
    ]
    outputs = [
        {
            "ground_truth": row.get("reference", row.get("ground_truth", "")),
            "reference_contexts": row.get("reference_contexts", row.get("contexts", []))
        } 
        for _, row in synth_df.iterrows()
    ]

    ls_client.create_examples(
        inputs=inputs,
        outputs=outputs,
        dataset_id=dataset.id,
    )
    print(f"🚀 Pushed {len(inputs)} synthetic Q&A examples to LangSmith Dataset '{DATASET_NAME}'.")
else:
    print(f"✅ Benchmark Dataset '{DATASET_NAME}' already exists on LangSmith.")

 Loaded 31 document pages/files from 'document_corpus'.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Benchmark Dataset 'Chunking_Strategy_Benchmark_v1' already exists on LangSmith.


### Dynamic Vector Store & RAG Target Pipeline Builder

In [4]:
# %% [markdown]
# ### Cell 3: Dynamic Vector Store & RAG Target Pipeline Builder

# %%
import shutil
import time
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

from langchain_huggingface import HuggingFaceEmbeddings

# Initialize local embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

def build_chunk_eval_pipeline(chunk_size: int, chunk_overlap: int, exp_name: str):
    """
    Builds an isolated ChromaDB vector store for a target chunk configuration
    and returns a LangSmith-compatible execution function.
    """
    exp_db_dir = os.path.join("data", f"chroma_exp_{exp_name}")
    
    # Clean previous experiment DB directory if it exists
    if os.path.exists(exp_db_dir):
        shutil.rmtree(exp_db_dir)
        
    # 1. Chunk documents with target parameters
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap
    )
    chunked_docs = splitter.split_documents(documents)
    
    # 2. Persist temporary experiment ChromaDB
    vectorstore = Chroma.from_documents(
        documents=chunked_docs,
        embedding=embeddings,
        persist_directory=exp_db_dir
    )
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    
    # 3. Target execution function (inputs: dict) -> outputs: dict
    def run_target_rag(inputs: dict) -> dict:
        question = inputs["question"]
        
        # Retrieve context chunks
        retrieved_docs = retriever.invoke(question)
        context_text = "\n\n".join([d.page_content for d in retrieved_docs])
        
        # Pause briefly to prevent API rate limits
        time.sleep(0.5)
        
        # Query Groq model via OpenAI API format
        response = groq_client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": f"Answer the user's question based strictly on the context below:\n\n{context_text}"},
                {"role": "user", "content": question}
            ]
        )
        
        answer = response.choices[0].message.content
        
        return {
            "output": answer,
            "retrieved_contexts": [d.page_content for d in retrieved_docs]
        }
        
    return run_target_rag

print("✅ Pipeline builder function initialized.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Pipeline builder function initialized.


### Custom LLM-as-a-Judge Evaluators

In [5]:
# %% [markdown]
# ### Cell 4: Custom LLM-as-a-Judge Evaluators for LangSmith

# %%
import json
from langsmith.evaluation import EvaluationResult, run_evaluator

@run_evaluator
def evaluate_context_recall(run, example) -> EvaluationResult:
    """Evaluates whether retrieved chunks contain ground truth facts."""
    ground_truth = example.outputs.get("ground_truth", "")
    contexts = run.outputs.get("retrieved_contexts", [])
    
    prompt = f"""Compare the ground truth answer with the retrieved contexts.
GROUND TRUTH: {ground_truth}
RETRIEVED CONTEXTS: {contexts}

Does the retrieved context contain the information required to answer the ground truth?
Return JSON ONLY: {{"score": 1.0 or 0.0, "reason": "short explanation"}}"""

    eval_response = groq_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    try:
        data = json.loads(eval_response.choices[0].message.content)
        return EvaluationResult(key="context_recall", score=float(data["score"]), comment=data.get("reason", ""))
    except Exception as e:
        return EvaluationResult(key="context_recall", score=0.0, comment=f"Parse error: {str(e)}")


@run_evaluator
def evaluate_faithfulness(run, example) -> EvaluationResult:
    """Evaluates whether generated response has hallucinations relative to retrieved chunks."""
    output = run.outputs.get("output", "")
    contexts = run.outputs.get("retrieved_contexts", [])
    
    prompt = f"""Check if the generated answer is strictly grounded in retrieved contexts.
GENERATED ANSWER: {output}
RETRIEVED CONTEXTS: {contexts}

Is the generated answer fully supported without any hallucination or outside information?
Return JSON ONLY: {{"score": 1.0 or 0.0, "reason": "short explanation"}}"""

    eval_response = groq_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    try:
        data = json.loads(eval_response.choices[0].message.content)
        return EvaluationResult(key="faithfulness", score=float(data["score"]), comment=data.get("reason", ""))
    except Exception as e:
        return EvaluationResult(key="faithfulness", score=0.0, comment=f"Parse error: {str(e)}")

print("✅ Custom evaluators configured.")

✅ Custom evaluators configured.


In [6]:
# %% [markdown]
# ### Cell 2: Synthetic Ground-Truth Dataset Generation (Ragas + Groq)

# %%
import sys
import types
import os
import glob

# ---------------------------------------------------------------------------
# 1. Compatibility Patch: Bypass Ragas legacy VertexAI import issue
# ---------------------------------------------------------------------------
try:
    import langchain_community.chat_models.vertexai
except ModuleNotFoundError:
    dummy_module = types.ModuleType("langchain_community.chat_models.vertexai")
    dummy_module.ChatVertexAI = type("ChatVertexAI", (object,), {})
    sys.modules["langchain_community.chat_models.vertexai"] = dummy_module

from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# ---------------------------------------------------------------------------
# 2. Load Document Corpus from 'document_corpus' folder
# ---------------------------------------------------------------------------
docs_folder = "document_corpus"
documents = []

if os.path.exists(docs_folder):
    for filepath in glob.glob(f"{docs_folder}/*.*"):
        if filepath.endswith(".pdf"):
            loader = PyPDFLoader(filepath)
            documents.extend(loader.load())
        elif filepath.endswith(".txt") or filepath.endswith(".md"):
            loader = TextLoader(filepath, encoding="utf-8")
            documents.extend(loader.load())
    print(f" Loaded {len(documents)} document pages/files from '{docs_folder}'.")
else:
    raise FileNotFoundError(f"❌ '{docs_folder}' folder does not exist. Create it and place your source documents inside.")

if len(documents) == 0:
    raise ValueError(f"❌ No .pdf, .txt, or .md files found inside '{docs_folder}'.")

 Loaded 31 document pages/files from 'document_corpus'.


### Experiment Execution Loop & Benchmarking

In [ ]:
# %% [markdown]
# ### Cell 5: Experiment Execution Loop & LangSmith Benchmarking

# %%
from langsmith.evaluation import evaluate

DATASET_NAME = "Chunking_Strategy_Benchmark_v1"

# Define strategy variants to benchmark
strategies = [
    {"name": "XLarge_2048", "chunk_size": 2048, "overlap": 300},
]

evaluators_new = [evaluate_context_recall, evaluate_faithfulness]

for strat in strategies:
    print(f"\n========================================================")
    print(f"🚀 Running Experiment: {strat['name']} (Size: {strat['chunk_size']}, Overlap: {strat['overlap']})")
    print(f"========================================================\n")
    
    # 1. Build temporary vector store & pipeline function
    target_fn = build_chunk_eval_pipeline(
        chunk_size=strat["chunk_size"],
        chunk_overlap=strat["overlap"],
        exp_name=strat["name"]
    )
    
    # 2. Run evaluation on LangSmith
    evaluate(
        target_fn,
        data=DATASET_NAME,
        evaluators=evaluators_new,
        experiment_prefix=f"chunk-{strat['name']}",
        metadata={
            "chunk_size": strat["chunk_size"],
            "chunk_overlap": strat["overlap"],
            "model": "llama-3.1-8b-instant",
            "embeddings": "all-MiniLM-L6-v2"
        }
    )

print("\n🎉 All experiments completed successfully!")
print("👉 Open your dashboard to view side-by-side results: https://smith.langchain.com/")


🚀 Running Experiment: Large_1024 (Size: 1024, Overlap: 200)

View the evaluation results for experiment: 'chunk-Large_1024-c804f46b' at:
https://apac.smith.langchain.com/o/de69f496-902d-4ca7-b209-fd22b9f10dad/datasets/be264044-814a-4de2-b9a5-0f1b82a11ffd/compare?selectedSessions=f7a89a46-5499-4dd3-9a68-4329be50d99b




0it [00:00, ?it/s]

Error running evaluator <DynamicRunEvaluator evaluate_context_recall> on run 01a11852-cc89-7ac2-bc04-5663e8e43f86: RateLimitError("Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3hxx4gqe73b5cjacazm9xda` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 197210, Requested 3085. Please try again in 2m7.44s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}")
Traceback (most recent call last):
  File "e:\5th semester lecture notes\supervisor_agent\.venv\Lib\site-packages\langsmith\evaluation\_runner.py", line 1723, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
        run=run,
        example=example,
        evaluator_run_id=evaluator_run_id,
    )
  File "e:\5th semester lecture notes\supervisor_agent\.venv\Lib\site-packages\langsmith\evaluation\evaluator.py", line 


🚀 Running Experiment: XLarge_2048 (Size: 2048, Overlap: 300)

View the evaluation results for experiment: 'chunk-XLarge_2048-d837516c' at:
https://apac.smith.langchain.com/o/de69f496-902d-4ca7-b209-fd22b9f10dad/datasets/be264044-814a-4de2-b9a5-0f1b82a11ffd/compare?selectedSessions=0eb8865e-e863-463f-abca-b779d45bd686




0it [00:00, ?it/s]

Error running target function: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3hxx4gqe73b5cjacazm9xda` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199611, Requested 3208. Please try again in 20m17.808s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Traceback (most recent call last):
  File "e:\5th semester lecture notes\supervisor_agent\.venv\Lib\site-packages\langsmith\evaluation\_runner.py", line 2062, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
    ~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\5th semester lecture notes\supervisor_agent\.venv\Lib\site-packages\langsmith\run_helpers.py", line 849, in wrapper
    function_result = run_container["context"].run(
        func, *args, **kwargs
    )
  File "C:\Users\MSI\AppData\Local\Temp\ipykernel_28848\2915187801.py", line 53, in


🎉 All experiments completed successfully!
👉 Open your dashboard to view side-by-side results: https://smith.langchain.com/


In [7]:
# %% [markdown]
# ### Cell 5: Rate-Paced Experiment Execution Loop (Single-Threaded)

# %%
import time
from langsmith.evaluation import evaluate

DATASET_NAME = "Chunking_Strategy_Benchmark_v1"
MODEL_NAME = "openai/gpt-oss-20b"

# Define strategy variants to benchmark
strategies = [
    {"name": "XLarge_2048", "chunk_size": 2048, "overlap": 300},
]

evaluators_new = [evaluate_context_recall, evaluate_faithfulness]

for strat in strategies:
    print(f"\n========================================================")
    print(f"🚀 Running Experiment: {strat['name']} (Size: {strat['chunk_size']}, Overlap: {strat['overlap']})")
    print(f"========================================================\n")
    
    # 1. Build temporary vector store & pipeline function
    target_fn = build_chunk_eval_pipeline(
        chunk_size=strat["chunk_size"],
        chunk_overlap=strat["overlap"],
        exp_name=strat["name"]
    )
    
    # 2. Run evaluation on LangSmith with strictly single-threaded execution
    evaluate(
        target_fn,
        data=DATASET_NAME,
        evaluators=evaluators_new,
        max_concurrency=1,  # Forces sequential execution to protect TPM / RPM limits
        experiment_prefix=f"chunk-{strat['name']}",
        metadata={
            "chunk_size": strat["chunk_size"],
            "chunk_overlap": strat["overlap"],
            "model": MODEL_NAME,
            "embeddings": "all-MiniLM-L6-v2"
        }
    )

print("\n🎉 All experiments completed successfully!")
print("👉 Open your dashboard to view side-by-side results: https://smith.langchain.com/")


🚀 Running Experiment: XLarge_2048 (Size: 2048, Overlap: 300)

View the evaluation results for experiment: 'chunk-XLarge_2048-85135a03' at:
https://apac.smith.langchain.com/o/de69f496-902d-4ca7-b209-fd22b9f10dad/datasets/be264044-814a-4de2-b9a5-0f1b82a11ffd/compare?selectedSessions=c6cad942-b1f0-4551-8505-d55856a0a814




0it [00:00, ?it/s]

Error running evaluator <DynamicRunEvaluator evaluate_context_recall> on run 01a11861-5125-7280-9b58-2311e3dfaad4: RateLimitError("Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3hxx4gqe73b5cjacazm9xda` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199220, Requested 1996. Please try again in 8m45.312s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}")
Traceback (most recent call last):
  File "e:\5th semester lecture notes\supervisor_agent\.venv\Lib\site-packages\langsmith\evaluation\_runner.py", line 1723, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
        run=run,
        example=example,
        evaluator_run_id=evaluator_run_id,
    )
  File "e:\5th semester lecture notes\supervisor_agent\.venv\Lib\site-packages\langsmith\evaluation\evaluator.py", lin


🎉 All experiments completed successfully!
👉 Open your dashboard to view side-by-side results: https://smith.langchain.com/
